# Banking Fraud Detection — Microsoft Fabric Real-Time Intelligence

This notebook demonstrates an **end-to-end banking fraud detection** workflow
built on top of **Microsoft Fabric Eventhouse (KQL Database)**.

## What you will learn
| # | Topic | Description |
|---|-------|-------------|
| 1 | Setup & Connection | Connect to Fabric Eventhouse via `azure-kusto-data` |
| 2 | Data Exploration | Inspect tables, row counts, distributions |
| 3 | Fraud Detection Functions | 10 KQL-based detection functions |
| 4 | Dashboard KPIs | Summary metrics & real-time alert feed |
| 5 | Investigation Workflow | Deep-dive into a flagged customer |
| 6 | Visualizations | Matplotlib charts for fraud analytics |
| 7 | Materialized Views | Pre-computed fast analytics |

## Prerequisites
- A **Microsoft Fabric workspace** with an **Eventhouse** provisioned
- A **KQL Database** named `BankingFraudDB` with the fraud-detection schema deployed
- Python 3.10+ with `azure-kusto-data`, `azure-identity`, `pandas`, `matplotlib`
- Azure credentials configured (e.g. `az login` or Managed Identity)

In [ ]:
# Install required packages (run once)
%pip install azure-kusto-data azure-identity pandas matplotlib --quiet

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.identity import DefaultAzureCredential

# ── Replace with your Eventhouse URI ────────────────────────────────────
cluster_uri = "https://<your-eventhouse>.z6.kusto.fabric.microsoft.com"
database    = "BankingFraudDB"

credential = DefaultAzureCredential()
kcsb = KustoConnectionStringBuilder.with_azure_token_credential(cluster_uri, credential)
client = KustoClient(kcsb)

print(f"Connected to {cluster_uri} / {database}")

In [ ]:
import pandas as pd

def run_kql(query: str, db: str = database) -> pd.DataFrame:
    """Execute a KQL query and return the result as a pandas DataFrame."""
    response = client.execute(db, query)
    primary = response.primary_results[0]
    columns = [col.column_name for col in primary.columns]
    rows = [row.to_list() for row in primary]
    return pd.DataFrame(rows, columns=columns)

print("Helper function ready.")

---
# 2 · Data Exploration

Discover the tables, row counts, and basic statistics of the
`BankingFraudDB` database.

In [ ]:
# Show all tables with details
df_tables = run_kql(".show tables details")
df_tables

In [ ]:
# Row counts per table
df_counts = run_kql("""
union
  (Customers       | count | extend Table = 'Customers'),
  (BankAccounts    | count | extend Table = 'BankAccounts'),
  (Merchants       | count | extend Table = 'Merchants'),
  (Transactions    | count | extend Table = 'Transactions')
| project Table, Count
""")
df_counts

In [ ]:
# Sample rows — Customers
run_kql("Customers | take 5")

In [ ]:
# Sample rows — BankAccounts
run_kql("BankAccounts | take 5")

In [ ]:
# Sample rows — Merchants
run_kql("Merchants | take 5")

In [ ]:
# Sample rows — Transactions
run_kql("Transactions | take 5")

In [ ]:
# Transaction amount distribution
df_stats = run_kql("""
Transactions
| summarize
    MinAmount   = min(Amount),
    AvgAmount   = round(avg(Amount), 2),
    MedianAmount = percentile(Amount, 50),
    P95Amount   = percentile(Amount, 95),
    MaxAmount   = max(Amount),
    TotalTxns   = count()
""")
df_stats

In [ ]:
# Transactions per channel
df_channel = run_kql("""
Transactions
| summarize Count = count() by Channel
| order by Count desc
""")
df_channel

In [ ]:
# Transactions per type
df_txn_type = run_kql("""
Transactions
| summarize Count = count() by TransactionType
| order by Count desc
""")
df_txn_type

---
# 3 · Fraud Detection Functions

Each subsection describes a KQL function, then executes it.

## 3.1 FraudDetection()
Multi-factor scoring engine that evaluates **6 rules** per transaction:
high amount, unusual hour, international mismatch, velocity, risky
merchant, and channel anomaly. Each rule adds to a cumulative score.

In [ ]:
df_fraud = run_kql("FraudDetection() | take 50")
print(f"Rows returned: {len(df_fraud)}")
df_fraud.head(10)

## 3.2 VelocityDetection()
Flags **rapid-fire transactions** where the same customer makes
multiple transactions within a **5-minute** window.

In [ ]:
df_velocity = run_kql("VelocityDetection() | take 50")
print(f"Rows returned: {len(df_velocity)}")
df_velocity.head(10)

## 3.3 CrossBorderFraud()
Detects **international transactions** and classifies them by risk
(domestic / low / medium / high / critical).

In [ ]:
df_cross = run_kql("CrossBorderFraud() | take 50")
print(f"Rows returned: {len(df_cross)}")
df_cross.head(10)

## 3.4 AmountAnomalyDetection()
Flags transactions where the amount is **5× or more** above the
customer’s historical average spend.

In [ ]:
df_amount = run_kql("AmountAnomalyDetection() | take 50")
print(f"Rows returned: {len(df_amount)}")
df_amount.head(10)

## 3.5 UnusualTimeDetection()
Flags transactions occurring during **12 AM – 6 AM**, an unusual
window for most banking activity.

In [ ]:
df_time = run_kql("UnusualTimeDetection() | take 50")
print(f"Rows returned: {len(df_time)}")
df_time.head(10)

## 3.6 AccountTakeoverDetection()
Detects potential account takeover when a customer has **10+ transactions
per hour** or operates from **3+ countries within 1 hour**.

In [ ]:
df_takeover = run_kql("AccountTakeoverDetection() | take 50")
print(f"Rows returned: {len(df_takeover)}")
df_takeover.head(10)

## 3.7 GeographicVelocityCheck()
Detects **impossible travel** — when a customer transacts from a
different country less than **4 hours** after the previous transaction.

In [ ]:
df_geo = run_kql("GeographicVelocityCheck() | take 50")
print(f"Rows returned: {len(df_geo)}")
df_geo.head(10)

## 3.8 MultiChannelSwitching()
Flags customers who switch between **3 or more channels** (ATM, online,
POS, mobile, etc.) within a **30-minute** window.

In [ ]:
df_channel_sw = run_kql("MultiChannelSwitching() | take 50")
print(f"Rows returned: {len(df_channel_sw)}")
df_channel_sw.head(10)

## 3.9 MerchantCategoryAnomaly()
Flags transactions in a merchant category where the amount is **5×+
above** the customer’s baseline spending in that category.

In [ ]:
df_merchant_anom = run_kql("MerchantCategoryAnomaly() | take 50")
print(f"Rows returned: {len(df_merchant_anom)}")
df_merchant_anom.head(10)

## 3.10 CustomerRiskProfile()
Generates a **composite risk score** per customer by aggregating fraud
signals, spending patterns, and behavioral anomalies.

In [ ]:
df_risk = run_kql("CustomerRiskProfile() | take 50")
print(f"Rows returned: {len(df_risk)}")
df_risk.head(10)

---
# 4 · Dashboard KPIs

Pre-built KQL functions that return KPI card values and a unified
real-time alert feed.

## 4.1 FraudSummaryKPIs()
Returns headline KPI values: total transactions, flagged count, fraud
rate, total amount at risk, etc.

In [ ]:
df_kpis = run_kql("FraudSummaryKPIs()")
df_kpis

## 4.2 RealTimeFraudDashboard()
Unified alert feed combining outputs from all detection functions,
ordered by severity and recency.

In [ ]:
df_dashboard = run_kql("RealTimeFraudDashboard() | take 100")
print(f"Alerts returned: {len(df_dashboard)}")
df_dashboard.head(20)

---
# 5 · Investigation Workflow

Deep-dive into a specific flagged customer to review their transaction
history, fraud flags, and risk profile.

In [ ]:
# Investigate customer CUST000595
customer_id = "CUST000595"

df_investigation = run_kql(f"FraudInvestigator('{customer_id}')")
print(f"Investigation records: {len(df_investigation)}")
df_investigation

In [ ]:
# Transaction history for the flagged customer
df_cust_txns = run_kql(f"""
Transactions
| where CustomerID == '{customer_id}'
| order by Timestamp desc
| take 50
""")
print(f"Recent transactions: {len(df_cust_txns)}")
df_cust_txns.head(20)

In [ ]:
# Fraud flags raised for the customer
df_cust_fraud = run_kql(f"""
FraudDetection()
| where CustomerID == '{customer_id}'
| order by FraudScore desc
""")
print(f"Fraud flags: {len(df_cust_fraud)}")
df_cust_fraud

In [ ]:
# Risk profile for the customer
df_cust_risk = run_kql(f"""
CustomerRiskProfile()
| where CustomerID == '{customer_id}'
""")
df_cust_risk

---
# 6 · Visualizations

Matplotlib charts for fraud analytics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 11})

## 6.1 Fraud Trend Over Time

In [ ]:
df_trend = run_kql("""
FraudDetection()
| summarize FraudCount = count() by bin(Timestamp, 1d)
| order by Timestamp asc
""")

fig, ax = plt.subplots()
ax.plot(pd.to_datetime(df_trend["Timestamp"]), df_trend["FraudCount"],
        marker="o", color="crimson")
ax.set_title("Daily Fraud Detections")
ax.set_xlabel("Date")
ax.set_ylabel("Count")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6.2 Fraud by Type (Pie Chart)

In [ ]:
df_type = run_kql("""
FraudDetection()
| summarize Count = count() by FraudType
| order by Count desc
""")

fig, ax = plt.subplots()
ax.pie(df_type["Count"], labels=df_type["FraudType"],
       autopct="%1.1f%%", startangle=140)
ax.set_title("Fraud Detections by Type")
plt.tight_layout()
plt.show()

## 6.3 Fraud by Channel (Bar Chart)

In [ ]:
df_chan = run_kql("""
FraudDetection()
| summarize Count = count() by Channel
| order by Count desc
""")

fig, ax = plt.subplots()
ax.bar(df_chan["Channel"], df_chan["Count"], color="steelblue")
ax.set_title("Fraud Detections by Channel")
ax.set_xlabel("Channel")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 6.4 Transaction Amount Distribution — Fraud vs Normal

In [ ]:
df_dist = run_kql("""
Transactions
| extend IsFraud = iif(FraudFlag == true, 'Fraud', 'Normal')
| project Amount, IsFraud
""")

fig, ax = plt.subplots()
for label, grp in df_dist.groupby("IsFraud"):
    ax.hist(grp["Amount"].astype(float), bins=50, alpha=0.6, label=label)
ax.set_title("Transaction Amount Distribution: Fraud vs Normal")
ax.set_xlabel("Amount")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()

## 6.5 Hourly Transaction Heatmap

In [ ]:
import numpy as np

df_hourly = run_kql("""
Transactions
| extend Hour = hourofday(Timestamp), DayOfWeek = dayofweek(Timestamp) / 1d
| summarize Count = count() by DayOfWeek = toint(DayOfWeek), Hour
| order by DayOfWeek asc, Hour asc
""")

pivot = df_hourly.pivot(index="DayOfWeek", columns="Hour", values="Count").fillna(0)
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(len(day_labels)))
ax.set_yticklabels(day_labels)
ax.set_xticks(range(24))
ax.set_xlabel("Hour of Day")
ax.set_title("Hourly Transaction Heatmap (Mon–Sun)")
fig.colorbar(im, ax=ax, label="Transaction Count")
plt.tight_layout()
plt.show()

---
# 7 · Materialized Views

Materialized views provide **pre-computed, incrementally updated**
aggregations that power fast dashboard queries.

## 7.1 mv_HourlyTransactionStats
Hourly transaction aggregates: count, total amount, average amount,
and fraud count per hour.

In [ ]:
df_mv_hourly = run_kql("""
mv_HourlyTransactionStats
| order by HourBucket desc
| take 48
""")
df_mv_hourly

## 7.2 mv_CustomerDailyRisk
Daily risk aggregates per customer: transaction count, total spend,
distinct countries, channels, and fraud flag count.

In [ ]:
df_mv_risk = run_kql("""
mv_CustomerDailyRisk
| order by DayBucket desc, FraudFlagCount desc
| take 50
""")
df_mv_risk

## 7.3 mv_MerchantRiskProfile
Merchant-level risk profile: total transactions, fraud count, fraud
rate, average transaction amount, and distinct customer count.

In [ ]:
df_mv_merchant = run_kql("""
mv_MerchantRiskProfile
| order by FraudRate desc
| take 50
""")
df_mv_merchant

---
## Summary

This notebook walked through the full **Banking Fraud Detection** workflow on
Microsoft Fabric Real-Time Intelligence:

1. **Connected** to the Eventhouse KQL Database
2. **Explored** tables and data distributions
3. **Executed 10 fraud detection functions** covering velocity, geography,
   amount anomalies, time-of-day, account takeover, and more
4. **Retrieved dashboard KPIs** and real-time alert feeds
5. **Investigated** a flagged customer end-to-end
6. **Visualized** fraud trends, distributions, and heatmaps
7. **Queried materialized views** for fast pre-computed analytics

> **Next steps:** Deploy these KQL functions in an Eventhouse, wire them to
> Fabric Real-Time Dashboard tiles, and configure Data Activator alerts for
> automated fraud notifications.